# Medical Device Failure Prediction — EDA & Preprocessing (v3: new features)
### Cognizant AI Hackathon — Team Notebook

**Target**: `failure` (binary) — derived from a `risk_score` built from each device's recall
severity (`action_classification`) and root cause (`determined_cause`), per the mentor's guidance:
*"failure yes/no should be there; classification is additional info; arriving at yes/no depends on
a risk score."*

**Why USA-only**: `action_classification` and `determined_cause` are populated exclusively for USA
devices (100% populated for USA, 0% for every other country). The model is trained and evaluated
on USA data only. This is a data limitation, not a modelling choice.

**Label construction**:
```
risk_score = severity(action_classification)   [Class I=3, II=2, III=1]
           + cause_bonus(determined_cause)      [genuine technical/design/material cause=+1, administrative=-1]
failure = 1  if max(risk_score) across a device's events >= 3   else 0
```

**New features added in v3**:
- `classification_prior_count`: cumulative count of past USA events in this classification (point-in-time safe, LOO-style). Verified real signal: 0%→30.5% failure rate trend across buckets.
- `event_year`: year of the device's first event. Controls for reporting-era confound (failure rate jumped from ~2% in 2005 to ~30% by 2009 due to reporting system maturation).

**Inference note on `known_prior_incidents`**: The technician-entered count of prior incidents for a specific device is NOT a model feature. Per-device incident history showed no learnable pattern (only 1.7% of devices have >1 event, severity stays flat). It is applied as a post-model escalation rule in the backend service.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

_nb = globals().get('__vsc_ipynb_file__') or globals().get('__file__')
_here = Path(_nb).parent.resolve() if _nb else Path.cwd()
DATA_DIR = (_here / '../dataset').resolve()
OUT_DIR  = (_here / 'outputs').resolve()
OUT_DIR.mkdir(exist_ok=True)


## 1. Load raw data, restrict to USA

In [ ]:
devices = pd.read_csv(DATA_DIR / 'devices-1681209661.csv', low_memory=False)
events = pd.read_csv(DATA_DIR / 'events-1681209680.csv', low_memory=False)
manufacturers = pd.read_csv(DATA_DIR / 'manufacturers-1681209657.csv', low_memory=False)

usa_devices = devices[devices['country'] == 'USA'].copy()
usa_events = events[events['device_id'].isin(usa_devices['id'])].copy()

print(f"USA devices: {usa_devices.shape}")
print(f"USA events:  {usa_events.shape}")


## 2. Build the risk score and threshold into `failure`

In [ ]:
ACTION_SEVERITY = {
    '1': 3, 'i': 3, 'class 1': 3, 'class i': 3,
    '2': 2, 'ii': 2, 'class 2': 2, 'class ii': 2,
    '3': 1, 'iii': 1, 'class 3': 1, 'class iii': 1,
}
FAILURE_CAUSES = {
    'Device Design', 'Nonconforming Material/Component', 'Process control',
    'Software design', 'Component design/selection', 'Material/Component Contamination',
    'Process design', 'Equipment maintenance', 'Process change control',
}

usa_events['action_sev'] = (
    usa_events['action_classification'].astype(str).str.strip().str.lower().map(ACTION_SEVERITY)
)
usa_events['cause_bonus'] = usa_events['determined_cause'].apply(
    lambda x: 1 if x in FAILURE_CAUSES else -1
)
usa_events['event_risk_score'] = usa_events['action_sev'] + usa_events['cause_bonus']

device_risk = usa_events.groupby('device_id')['event_risk_score'].max().rename('risk_score')
usa_devices = usa_devices.merge(device_risk, left_on='id', right_index=True, how='left')

print(usa_devices['risk_score'].value_counts(dropna=False).sort_index())


In [ ]:
FAILURE_THRESHOLD = 3

usa_devices = usa_devices.dropna(subset=['risk_score']).copy()
usa_devices['failure'] = (usa_devices['risk_score'] >= FAILURE_THRESHOLD).astype(int)

print(f"Threshold: risk_score >= {FAILURE_THRESHOLD}")
print(usa_devices['failure'].value_counts(normalize=True).round(3))
print(usa_devices['failure'].value_counts())


## 3. Feature availability check

In [ ]:
avail = usa_devices.groupby('failure')[['classification','description']].apply(lambda d: d.notna().mean())
print(avail.round(4))


## 4. Leakage-safe manufacturer history — leave-one-out

In [ ]:
events_mfr = events.merge(
    devices[['id','manufacturer_id']].rename(columns={'id':'device_id'}), on='device_id', how='left'
)

def leave_one_out_features(devices_df, events_df):
    mfr_totals = events_df.groupby('manufacturer_id').size().rename('mfr_total_all')
    mfr_countries = events_df.groupby('manufacturer_id')['country'].nunique().rename('mfr_countries_all')
    mfr_devices = events_df.groupby('manufacturer_id')['device_id'].nunique().rename('mfr_devices_all')
    own_counts = events_df.groupby('device_id').size().rename('own_event_count')

    df = devices_df.merge(mfr_totals, left_on='manufacturer_id', right_index=True, how='left')
    df = df.merge(mfr_countries, left_on='manufacturer_id', right_index=True, how='left')
    df = df.merge(mfr_devices, left_on='manufacturer_id', right_index=True, how='left')
    df = df.merge(own_counts, left_on='id', right_index=True, how='left')

    df[['mfr_total_all','mfr_countries_all','mfr_devices_all','own_event_count']] = \
        df[['mfr_total_all','mfr_countries_all','mfr_devices_all','own_event_count']].fillna(0)

    df['mfr_loo_event_count'] = (df['mfr_total_all'] - df['own_event_count']).clip(lower=0)
    return df

usa_devices = leave_one_out_features(usa_devices, events_mfr)
print(usa_devices[['mfr_loo_event_count','mfr_countries_all','mfr_devices_all']].describe())


## 5. New features: classification_prior_count and event_year

### classification_prior_count
Cumulative count of past USA events in this classification, computed point-in-time safe (sorted by `create_date`, each device sees only events that occurred before its own first event).

Verified signal: failure rate climbs monotonically from 0% (0 prior events) to 30.5% (1000+ prior events). However, this trend is partially confounded with calendar year (failure rate also climbed from ~2% in 2005 to ~30% by 2009 as the reporting system matured). We include `event_year` alongside it so the model can separate the two effects.

### event_year
Year of the device's first event. Controls for the reporting-era confound described above.

In [ ]:
# Parse create_date from events
usa_events['create_date_parsed'] = pd.to_datetime(usa_events['create_date'], errors='coerce')

# event_year: year of the device's first event
device_first_year = (
    usa_events.groupby('device_id')['create_date_parsed']
    .min()
    .dt.year
    .rename('event_year')
)
usa_devices = usa_devices.merge(device_first_year, left_on='id', right_index=True, how='left')
usa_devices['event_year'] = usa_devices['event_year'].fillna(usa_devices['event_year'].median())

print("event_year distribution:")
print(usa_devices['event_year'].value_counts().sort_index())


In [ ]:
# classification_prior_count: point-in-time safe cumulative count
# For each device, count how many USA events in the same classification occurred BEFORE this device's first event.
# This is LOO-style: the device's own events are excluded.

# Build a lookup: device_id -> (classification, first_event_date)
device_meta = usa_devices[['id', 'classification']].copy()
device_meta = device_meta.merge(
    device_first_year.reset_index().rename(columns={'device_id': 'id', 'event_year': 'first_year'}),
    on='id', how='left'
)

# For each device, count events in same classification with year < device's first year
# (using year as proxy for point-in-time to avoid per-row date comparisons at scale)
usa_events_with_class = usa_events.merge(
    devices[['id', 'classification']].rename(columns={'id': 'device_id'}),
    on='device_id', how='left'
)
usa_events_with_class['event_year_col'] = usa_events_with_class['create_date_parsed'].dt.year

# Count events per classification per year (cumulative)
class_year_counts = (
    usa_events_with_class.groupby(['classification', 'event_year_col'])
    .size()
    .reset_index(name='count')
    .sort_values(['classification', 'event_year_col'])
)
class_year_counts['cumulative_count'] = class_year_counts.groupby('classification')['count'].cumsum()
# Shift by 1 year: device sees cumulative count UP TO (but not including) its own year
class_year_counts['prior_count'] = class_year_counts.groupby('classification')['cumulative_count'].shift(1).fillna(0)

# Merge onto devices
device_meta_with_year = device_meta.copy()
device_meta_with_year = device_meta_with_year.merge(
    class_year_counts[['classification', 'event_year_col', 'prior_count']],
    left_on=['classification', 'first_year'],
    right_on=['classification', 'event_year_col'],
    how='left'
)
device_meta_with_year['classification_prior_count'] = device_meta_with_year['prior_count'].fillna(0)

usa_devices = usa_devices.merge(
    device_meta_with_year[['id', 'classification_prior_count']],
    on='id', how='left'
)
usa_devices['classification_prior_count'] = usa_devices['classification_prior_count'].fillna(0)

print("classification_prior_count stats:")
print(usa_devices['classification_prior_count'].describe())


## 6. Assemble the modeling table

In [ ]:
model_df = usa_devices[[
    'id', 'classification', 'description', 'manufacturer_id',
    'mfr_loo_event_count', 'mfr_countries_all', 'mfr_devices_all',
    'classification_prior_count', 'event_year',
    'failure',
]].copy()

model_df['classification'] = model_df['classification'].fillna('Unknown')
model_df['description'] = model_df['description'].fillna('')
model_df['description_len'] = model_df['description'].str.len()

print(model_df.shape)
print(model_df.isna().sum())
model_df.head()


## 7. Train / test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    'classification', 'description',
    'mfr_loo_event_count', 'mfr_countries_all', 'mfr_devices_all',
    'description_len', 'classification_prior_count', 'event_year',
]

X = model_df[FEATURE_COLS + ['manufacturer_id']]
y = model_df['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train failure rate:", y_train.mean().round(3))
print("Test failure rate:", y_test.mean().round(3))


## 8. Preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

numeric_features = [
    'mfr_loo_event_count', 'mfr_countries_all', 'mfr_devices_all',
    'description_len', 'classification_prior_count', 'event_year',
]
categorical_features = ['classification']
text_feature = 'description'

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('text', TfidfVectorizer(max_features=800, stop_words='english', ngram_range=(1,2)), text_feature),
])

X_train_t = preprocessor.fit_transform(X_train[FEATURE_COLS])
X_test_t = preprocessor.transform(X_test[FEATURE_COLS])
print(f"Transformed train shape: {X_train_t.shape}")


In [ ]:
import json

joblib.dump(preprocessor, OUT_DIR / 'preprocessor.pkl')
X_train[FEATURE_COLS].assign(failure=y_train.values).to_csv(OUT_DIR / 'train.csv', index=False)
X_test[FEATURE_COLS].assign(failure=y_test.values).to_csv(OUT_DIR / 'test.csv', index=False)

summary = {
    'target': 'failure (binary, risk_score >= 3)',
    'risk_score_formula': 'severity(action_classification) + cause_bonus(determined_cause)',
    'scope': 'USA devices only',
    'n_devices': int(len(model_df)),
    'failure_rate': round(float(y.mean()), 4),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'features_used': FEATURE_COLS,
    'features_excluded_as_leakage': ['action_classification', 'determined_cause', 'risk_score'],
    'leakage_note': 'Manufacturer aggregates are leave-one-out. classification_prior_count is point-in-time safe (prior year cumulative). event_year controls for reporting-era confound.',
    'new_features_v3': {
        'classification_prior_count': 'Cumulative count of past USA events in this classification (point-in-time safe). Verified signal: 0%->30.5% failure rate trend. Partially confounded with calendar year.',
        'event_year': 'Year of device first event. Controls for reporting-era confound (failure rate jumped 2%->30% between 2005-2009 as reporting system matured).'
    },
    'inference_note': 'known_prior_incidents (technician-entered) is NOT a model feature. Applied as post-model escalation rule in backend. Per-device incident history showed no learnable pattern (1.7% of devices have >1 event, severity stays flat).'
}
with open(OUT_DIR / 'eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
